# Litigi per-user analysis

Example: https://scikit-learn.org/stable/auto_examples/text/plot_document_clustering.html#sphx-glr-auto-examples-text-plot-document-clustering-py

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
from functions.utilities import preprocess
from stop_words import get_stop_words
from time import time
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Normalizer
from sklearn.decomposition import PCA

In [ ]:
stop = get_stop_words('italian')


In [ ]:
filename='processed_data\itigi_comments_2024.parqet'
litigi=pd.read_parquet(filename).drop_duplicates(subset=['author','body','id', 'created_utc'])
litigi['created_dt']=(pd.to_datetime(litigi['created_utc'],unit='s'))
litigi['body_preprocessed']=litigi['body'].apply(lambda x: preprocess(x))

# extract the author of the parent comment (if applicable)
id_to_author = litigi.set_index('id')['author'].to_dict()
litigi['parent_author'] = litigi['parent_id'].str.split('_').str[1].map(id_to_author)

litigi.head()

# Comments per user

In [ ]:
top_users=litigi.groupby('author').size().sort_values(ascending=False).reset_index().rename(columns={0:'comments'})
top_users=top_users[top_users['author'] != '[deleted]']
top_users

# Hourly posting habits of top X users

In [ ]:
X=30
author_list = top_users['author'][:X]


plt.figure(figsize=(19, 10))

for author in author_list:
    sns.kdeplot(pd.to_datetime(litigi[litigi['author'] == author]['created_utc'], unit='s').dt.hour,label=author)
    #histograms.append(author_hist)

plt.xlabel('hour')
plt.ylabel('Frequency')
plt.legend(loc='upper right')
plt.show()

In [ ]:
import plotly.graph_objects as go
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde
import plotly.express as px
from scipy.spatial.distance import jensenshannon

In [ ]:
author_densities={}
X = 100
author_list = top_users['author'][:X]

for author in author_list:
    # Extract hours for current author
    author_hours = pd.to_datetime(litigi[litigi['author'] == author]['created_utc'], unit='s').dt.hour
    
    # Create mirrored data for periodic boundary conditions
    mirrored_hours = np.concatenate([author_hours - 24, author_hours, author_hours + 24])
    
    # Calculate KDE with mirrored data
    kde = gaussian_kde(mirrored_hours,bw_method=.05)
    x_grid = np.linspace(0, 24, 1000)
    density = kde.evaluate(x_grid) * 3  # Multiply by 3 to account for mirrored data
    author_densities[author] = density

In [ ]:
fig = go.Figure()

for author in author_list:
    density = author_densities[author]
    
    # Add trace to plot
    fig.add_trace(go.Scatter(
        x=x_grid,
        y=density,
        mode='lines',
        name=author,
        line=dict(width=1)
    ))

# Configure plot layout
fig.update_layout(
    title='Activity Distribution with Periodic Boundary Conditions',
    xaxis=dict(
        title='Hour of Day',
        range=[0, 24],
        dtick=2
    ),
    yaxis=dict(title='Density'),
    legend=dict(title='Authors'),
    width=1400,
    height=800,
    template='plotly_white'
)

fig.show()

In [ ]:
# Create a similarity matrix using Jensen-Shannon divergence
n_authors = len(author_list)
similarity_matrix = np.zeros((n_authors, n_authors))

for i, author1 in enumerate(author_list):
    for j, author2 in enumerate(author_list):
        # Compute Jensen-Shannon divergence (symmetric and bounded)
        js_divergence = jensenshannon(author_densities[author1], author_densities[author2])
        # Convert to similarity (higher values mean more similar)
        similarity_matrix[i, j] = 1 - js_divergence

# Create a DataFrame for the similarity matrix
similarity_df = pd.DataFrame(similarity_matrix, index=author_list, columns=author_list)

# Plot the similarity matrix as a heatmap
fig = go.Figure(data=go.Heatmap(
    z=similarity_df.values,
    x=similarity_df.columns,
    y=similarity_df.index,
    colorscale='Viridis',
    colorbar=dict(title='Similarity')
))
size=1200

fig.update_layout(
    title='Author Posting Habit Similarity Matrix',
    xaxis=dict(title='Authors'),
    yaxis=dict(title='Authors'),
    width=size,
    height=size,
    template='plotly_white'
)

fig.show()

In [ ]:
from scipy.spatial import distance
import scipy.sparse as scisp
from sklearn.preprocessing import normalize
import matplotlib

def ComputeLaplacian(X=None,rbf_p=1.,S_m=None,pseudo=True):
    """Compute laplacian matrix
    
    Returns Normalized laplacian, similarity matrix, degree matrix

    Args:
        X (_type_): _description_
        rbf_p (_type_, optional): _description_. Defaults to 1..
        S_m (_type_, optional): _description_. Defaults to None.
        pseudo (bool, optional):  compute speuso laplacian (1-L) instead of L. Defaults to True.

    Returns:
        _type_: Normalized laplacian, similarity matrix, degree matrix
    """    
    
    if S_m is None:
        #K=rbf_kernel(X, gamma=rbf_p)
        S_m=np.exp(-rbf_p*distance.cdist(X,X,metric='sqeuclidean')) # or rbf_kernel(X, gamma=rbf_p) , which is more performant?
    D=S_m.sum(axis=1) # degree matrix
    D_n=np.sqrt(1/D)
    M=np.multiply(D_n[np.newaxis,:],np.multiply(S_m,D_n[:,np.newaxis])) # normalized pseudo laplacian
    if not pseudo:
        M=np.identity(M.shape[0])-M

    return M,S_m,D

def SpectralEmbedding(X,rbf_p=1.,S_m=None,threshold=None,pseudo=None,k=10):
    """Returns the sepctral embedding of a dataset"""
    M,_,_=ComputeLaplacian(X,rbf_p,S_m,pseudo=pseudo)
    if threshold is not None:
        eivals,eivecs=scisp.linalg.eigsh(np.where(M < threshold, 0, M),k=k)
    else:
        eivals,eivecs=scisp.linalg.eigsh(M,k=k)
    return eivals,eivecs



In [ ]:
threshold=1e-15
k=20
M,K,_=ComputeLaplacian(S_m=similarity_matrix,rbf_p=10)
eivals,eivecs=scisp.linalg.eigsh(np.where(M < threshold, 0, M),k=k)

In [ ]:
eivec_flipped=np.flip(eivecs,1)
n_components=k
fig,axes=plt.subplots(1,1,squeeze=False,figsize=(20,7),sharex=True)
for i in range(n_components):
    #plt.figure(figsize=(20,5))
    #ax=plt.subplot()
    ax=axes[0][0]
    ax.scatter(range(len(eivec_flipped[:,1])),eivec_flipped[eivec_flipped[:,i].argsort()][:,i],s=4,label=f"{i+1}th eigenvector",alpha=.5)
plt.legend()
llim=0.05
#plt.ylim(top=llim,bottom=-llim)
plt.title("Sorted eigenvectors")
plt.show()

In [ ]:
vec=np.zeros(len(author_list))
k_max=k
fig,axes=plt.subplots(k_max,1,squeeze=False,figsize=(20,15),sharex=True)

for i in range(k_max):
    n_eigenvectors=i+1
    vec=np.sum(normalize(eivecs[:,-n_eigenvectors:],axis=1),axis=1)
    ax=axes[i][0]
    ax.scatter(range(len(author_list)),vec[vec.argsort()],s=2,label=f"Sum, ncomp = {n_eigenvectors}")
    ax.legend()
    ax.grid()

#ax.set_title(f"{i} eigenvector")

#plt.ylim(top=0.2,bottom=-.2)
plt.suptitle("Sorted eigenvectors")
plt.show()

In [ ]:
n=2
vec=np.zeros(len(author_list))
vec=np.sum(normalize(eivecs[:,-n:],axis=1),axis=1)
fig, ax = plt.subplots(figsize=(12,12))
index_rearranged=vec.argsort()

In [ ]:
# Plot the similarity matrix as a heatmap

reindexed_author_list=list(author_list.to_numpy()[index_rearranged])

similarity_df = pd.DataFrame(similarity_matrix[index_rearranged][:, index_rearranged], index=reindexed_author_list, columns=reindexed_author_list)

fig = go.Figure(data=go.Heatmap(
    z=similarity_df.values,
    x=similarity_df.columns,
    y=similarity_df.index,
    colorscale='Viridis',
    zmax=1,
    colorbar=dict(title='Similarity')
))

fig.update_layout(
    title='Author Posting Habit Similarity Matrix',
    xaxis=dict(title='Authors'),
    yaxis=dict(title='Authors'),
    width=1200,
    height=1200,
    template='plotly_white'
)

fig.show()

In [ ]:
litigi['created_hour']=litigi['created_dt'].dt.hour
litigi

In [ ]:

grouped_by_user=litigi[litigi['author'].isin(author_list)].groupby('author')


In [ ]:
hour=litigi[litigi['author'].isin(author_list)].groupby(['author','created_hour']).count()
user_comments_binned=hour[['body']].reset_index().pivot_table(index='author',columns='created_hour',aggfunc='sum').reset_index().fillna(0)
user_comments_binned

In [ ]:
user_comments=grouped_by_user['body_preprocessed'].agg(lambda x: ' '.join(x)).reset_index()
users=list(user_comments['author'])

In [ ]:
t0 = time()
vectorizer = TfidfVectorizer(stop_words=stop,max_features=1000)
X_tfidf = vectorizer.fit_transform(user_comments['body_preprocessed'])
print(f"vectorization done in {time() - t0:.3f} s")
print(f"n_samples: {X_tfidf.shape[0]}, n_features: {X_tfidf.shape[1]}")

In [ ]:
X_tfidf

In [ ]:
labels=user_comments['author']

In [ ]:



lsa = make_pipeline(TruncatedSVD(n_components=100), Normalizer(copy=False))
t0 = time()
X_lsa = lsa.fit_transform(X_tfidf)
explained_variance = lsa[0].explained_variance_ratio_.sum()

print(f"LSA done in {time() - t0:.3f} s")
print(f"Explained variance of the SVD step: {explained_variance * 100:.1f}%")

In [ ]:
L,Sm,D=ComputeLaplacian(X_lsa)
Sm

In [ ]:
# Plot the similarity matrix as a heatmap


similarity_df = pd.DataFrame(Sm, index=users, columns=users)

fig = go.Figure(data=go.Heatmap(
    z=similarity_df.values,
    x=similarity_df.columns,
    y=similarity_df.index,
    colorscale='Viridis',
    zmax=1,
    colorbar=dict(title='Similarity')
))

fig.update_layout(
    title='Author Posting Habit Similarity Matrix',
    xaxis=dict(title='Authors'),
    yaxis=dict(title='Authors'),
    width=1200,
    height=1200,
    template='plotly_white'
)

fig.show()

In [ ]:
tfidf_similarity = pd.DataFrame(Sm, index=users, columns=users)
posting_time_similarity = pd.DataFrame(similarity_matrix, index=users, columns=users)
tfidf_similarity_aligned = tfidf_similarity.reindex_like(posting_time_similarity).fillna(0)

In [ ]:
alpha=.2
full_matrix=alpha*posting_time_similarity.values+(1-alpha)*tfidf_similarity_aligned.values
full_matrix=full_matrix/full_matrix.max()
full_matrix

In [ ]:
# Plot the similarity matrix as a heatmap


similarity_df =pd.DataFrame(full_matrix,index=posting_time_similarity.index,columns=posting_time_similarity.columns)

fig = go.Figure(data=go.Heatmap(
    z=similarity_df.values,
    x=similarity_df.columns,
    y=similarity_df.index,
    colorscale='Viridis',
    zmax=1,
    colorbar=dict(title='Similarity')
))

fig.update_layout(
    title='Author Posting Habit Similarity Matrix',
    xaxis=dict(title='Authors'),
    yaxis=dict(title='Authors'),
    width=1200,
    height=1200,
    template='plotly_white'
)

fig.show()

In [ ]:
from scipy.cluster import hierarchy
# Given similarity matrix S
D = 1 - full_matrix  # Convert to dissimilarity

# Perform hierarchical clustering
Z = hierarchy.linkage(D, method='average')

# Optimize leaf order
optimal_Z = hierarchy.optimal_leaf_ordering(Z, D)

# Get permutation order
optimal_order = hierarchy.leaves_list(optimal_Z)

reindexed_author_list=list(similarity_df.columns.to_numpy()[optimal_order])

similarity_df = pd.DataFrame(full_matrix[optimal_order][:, optimal_order], index=reindexed_author_list, columns=reindexed_author_list)

# Plot the similarity matrix as a heatmap
fig = go.Figure(data=go.Heatmap(
    z=similarity_df.values,
    x=similarity_df.columns,
    y=similarity_df.index,
    colorscale='Viridis',
    zmax=1,
    colorbar=dict(title='Similarity')
))

fig.update_layout(
    title='Author Posting Habit Similarity Matrix',
    xaxis=dict(title='Authors'),
    yaxis=dict(title='Authors'),
    width=1200,
    height=1200,
    template='plotly_white'
)

fig.show()

In [ ]:

tfidf_similarity_aligned

In [ ]:
pca=make_pipeline(lsa,PCA(n_components=2))
X_pca = pca.fit_transform(X_tfidf)
X_pca

In [ ]:
px.scatter(x=X_pca[:,0],y=X_pca[:,1],labels=labels,color=labels)

In [ ]:
user_comments

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import numpy as np
# Vectorize text
vectorizer = CountVectorizer(stop_words=stop)
X = vectorizer.fit_transform(litigi[:10000]['body_preprocessed'])

# Apply LDA
lda = LatentDirichletAllocation(n_components=10, random_state=42)
lda.fit(X)

# Display topics
topics = np.argsort(lda.components_, axis=1)[:, -5:]
print("Top words per topic:")
for i, topic in enumerate(topics):
    print(f"Topic {i}: {[vectorizer.get_feature_names_out()[j] for j in topic]}")